# Vestibular-BR — Avaliação com vLLM (Kaggle)

**Configuração do notebook (painel lateral):** Accelerator = **GPU T4 x2**, Internet = **On**,
Secrets: `HF_TOKEN`.

O que este notebook faz:
1. Avalia a família Qwen3.5 (0.8B → 9B) no conjunto de teste → **curva de escala**
2. Compara **imagem real vs. descrição vs. sem imagem** no ENEM → **pergunta de pesquisa 2**
3. Roda o **teste de contaminação** (alternativas embaralhadas) → **pergunta de pesquisa 3**
4. Mede a taxa de acerto do modelo base no treino (filtro de dificuldade do GRPO)
5. Avalia os modelos treinados (SFT, GRPO, SFT+GRPO) → **pergunta de pesquisa 1**

No fim, baixe `resultados.zip` e copie `results/runs/` para o seu repositório.

In [ ]:
# >>> TROQUE pelo seu usuário do GitHub e do Hugging Face <<<
GITHUB_REPO = "https://github.com/SEU_USUARIO/vestibular-br.git"
HF_USER = "SEU_USUARIO_HF"

import os
from kaggle_secrets import UserSecretsClient  # Add-ons > Secrets: crie HF_TOKEN (e GEMINI_API_KEY se quiser)
secrets = UserSecretsClient()
for k in ["HF_TOKEN", "GEMINI_API_KEY"]:
    try:
        os.environ[k] = secrets.get_secret(k)
    except Exception:
        print(f"secret {k} não configurado (ok se não for usar)")

In [ ]:
!git clone -q $GITHUB_REPO repo
%cd repo
!pip install -q -e . vllm
!vbr data

## 1. Curva de escala — modelos base (modo descrição)

In [ ]:
for m in ["Qwen/Qwen3.5-0.8B", "Qwen/Qwen3.5-2B", "Qwen/Qwen3.5-4B"]:
    !vbr eval -b test -m vllm:{m}
# 9B precisa das duas T4:
!VLLM_TP=2 vbr eval -b test -m vllm:Qwen/Qwen3.5-9B

## 2. Imagem vs. descrição vs. sem imagem (ENEM 2022–2024)

In [ ]:
for mode in ["image", "blind"]:
    !vbr eval -b enem -m vllm:Qwen/Qwen3.5-2B --mode {mode}
    !VLLM_TP=2 vbr eval -b enem -m vllm:Qwen/Qwen3.5-9B --mode {mode}

## 3. Contaminação — alternativas embaralhadas
Também vale rodar em provas ANTIGAS do BLUEX (que estão na internet há anos): `-b bluex`.

In [ ]:
!VLLM_TP=2 vbr contamination -b bluex -m vllm:Qwen/Qwen3.5-9B --seeds 1 --seeds 2

## 4. Filtro de dificuldade para o GRPO
4 amostras por questão de treino. Questões com 0 < acertos < 4 são as que mais ensinam no GRPO.

In [ ]:
!vbr eval -b train -m vllm:Qwen/Qwen3.5-2B -n 4 --temperature 1.0 --max-tokens 1024 --tag pass4

## 5. Modelos treinados (rode depois do notebook de treino)

In [ ]:
for tag in ["sft", "grpo", "sft-grpo"]:
    !vbr eval -b test -m vllm:Qwen/Qwen3.5-2B --lora {HF_USER}/vestibular-br-2b-{tag} --tag {tag}

In [ ]:
!vbr report
!zip -qr /kaggle/working/resultados.zip results
print("Baixe resultados.zip na aba Output")